# PC Offload Colab Bootstrap

Use this notebook as a remote VM extension of the local machine. It mounts Google Drive, syncs the bootstrap repo, and gives you a safe place to run heavy work without loading local RAM or disk.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import os
import subprocess

REPO_URL = 'https://github.com/israelrealivazquez-lang/pc-speed-cloud-bootstrap.git'
WORKDIR = Path('/content/pc-speed-cloud-bootstrap')
DRIVE_MOUNT = Path('/content/drive')
PRIMARY_DRIVE_ROOT = DRIVE_MOUNT / 'MyDrive'
ROOT_NAME = 'Antigravity_Cloud_Migration'
OFFLOAD_NAMES = [
    'Offload_From_Dropbox_2026-04-19',
    'Offload_From_Profile_2026-04-19',
    'Offload_From_OneDrive_2026-04-19'
]

def find_named_directories(search_root: Path, names, max_depth: int = 5):
    wanted = set(names)
    results = {name: [] for name in wanted}
    for root, dirs, _ in os.walk(search_root):
        root_path = Path(root)
        try:
            depth = len(root_path.relative_to(search_root).parts)
        except ValueError:
            continue
        if depth > max_depth:
            dirs[:] = []
            continue
        for directory in dirs:
            if directory in wanted:
                results[directory].append(root_path / directory)
    return results

if WORKDIR.exists():
    subprocess.run(['git', '-C', str(WORKDIR), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(WORKDIR)], check=True)

search_results = find_named_directories(DRIVE_MOUNT, [ROOT_NAME] + OFFLOAD_NAMES, max_depth=5)
candidate_roots = [PRIMARY_DRIVE_ROOT / ROOT_NAME]
candidate_roots.extend(search_results.get(ROOT_NAME, []))
existing_roots = [candidate for candidate in candidate_roots if candidate.exists()]
DRIVE_ROOT = existing_roots[0] if existing_roots else candidate_roots[0]
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

TARGET_PATHS = {}
for name in OFFLOAD_NAMES:
    candidates = [DRIVE_ROOT / name]
    candidates.extend(search_results.get(name, []))
    TARGET_PATHS[name] = next((candidate for candidate in candidates if candidate.exists()), candidates[0])

print('Repo ready at', WORKDIR)
print('Drive root:', DRIVE_ROOT)
for name, path in TARGET_PATHS.items():
    print(f'Target[{name}] -> {path}')

In [ ]:
def describe_tree(path: Path):
    total = 0
    count = 0
    for root, _, files in os.walk(path):
        for name in files:
            candidate = Path(root) / name
            try:
                total += candidate.stat().st_size
                count += 1
            except OSError:
                pass
    return count, total

for name, target in TARGET_PATHS.items():
    if target.exists():
        count, total = describe_tree(target)
        print(f'{target}: {count} files, {round(total / 1e9, 3)} GB')
    else:
        print(f'{target}: missing')

In [ ]:
from pathlib import Path
import shutil

def resolve_drive_source(relative_path: str):
    requested = Path(relative_path)
    direct = DRIVE_ROOT / requested
    if direct.exists():
        return direct
    fallback = TARGET_PATHS.get(requested.name)
    if fallback and fallback.exists():
        return fallback
    return direct

def stage_from_drive(relative_path: str, local_root: str = '/content/work'):
    src = resolve_drive_source(relative_path)
    dst = Path(local_root) / Path(relative_path).name
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists():
        if dst.is_dir():
            shutil.rmtree(dst)
        else:
            dst.unlink()
    if not src.exists():
        raise FileNotFoundError(f'Drive source not found: {src}')
    if src.is_dir():
        shutil.copytree(src, dst)
    else:
        shutil.copy2(src, dst)
    return dst

def persist_to_drive(local_path: str, relative_path: str):
    src = Path(local_path)
    dst = DRIVE_ROOT / relative_path
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists():
        if dst.is_dir():
            shutil.rmtree(dst)
        else:
            dst.unlink()
    if src.is_dir():
        shutil.copytree(src, dst)
    else:
        shutil.copy2(src, dst)
    return dst

print('Helpers loaded: stage_from_drive(), persist_to_drive(), resolve_drive_source()')

## Suggested usage

1. Mount Drive.
2. Pull the latest repo version.
3. Let the notebook discover the offload root for the currently mounted Google account.
4. Stage a large folder from Drive into `/content/work`.
5. Run heavy processing in Colab.
6. Persist results back into the detected `Antigravity_Cloud_Migration` root.